# Scoring
> Score a sampling campaign against the ground truth

Work in progress: experiments for scoring campaigns and comparing sampling strategies, following `scoring-plan.md` (#12).

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance
import geopandas as gpd
from fastcore.test import *
from fastcore.utils import store_attr, patch

from trufl.utils import gridder, zonal_values
from trufl.sampler import Sampler, rank_to_sample
from trufl.collector import DataCollector
from trufl.callbacks import State, MaxCB, MinCB, StdCB, CountCB, MoranICB, PriorCB
from trufl.optimizer import Optimizer

One campaign to experiment on: a 10 x 10 grid over the simulated raster, with between 0 and 9 samples per unit, so some units have none:

In [ ]:
fname_raster = 'files/ground-truth-01-4326-simulated.tif'
gdf_grid = gridder(fname_raster, nrows=10, ncols=10)
n = np.random.default_rng(0).integers(0, 10, size=len(gdf_grid))
samples = DataCollector(fname_raster).collect(Sampler(gdf_grid).sample(n, method='uniform', rng=0))
samples.head()

`unit_stats` compares each unit's samples with the raster cells inside it. Units without samples have a count of 0 and `NaN` for the measured values and the distance:

In [ ]:
def unit_stats(
    samples:gpd.GeoDataFrame, # Measurements indexed by `loc_id`, with a `value` column
    gdf_units:gpd.GeoDataFrame, # Units indexed by `loc_id`
    fname_raster:str, # Ground-truth raster
) -> pd.DataFrame: # One row per unit
    "True and measured mean and maximum, sample count and Wasserstein distance of each unit."
    rows = {}
    for loc_id, geom in gdf_units.geometry.items():
        true = zonal_values(fname_raster, geom)
        meas = samples.value[samples.index == loc_id].values
        rows[loc_id] = dict(true_mean=true.mean(), true_max=true.max(),
                            meas_mean=meas.mean() if len(meas) else np.nan, meas_max=meas.max() if len(meas) else np.nan,
                            n=len(meas), wasserstein=wasserstein_distance(true, meas) if len(meas) else np.nan)
    return pd.DataFrame.from_dict(rows, orient='index').rename_axis('loc_id')

In [ ]:
df = unit_stats(samples, gdf_grid, fname_raster)
test_eq(df.n.sum(), len(samples))
df.head()

## Campaigns

A `Campaign` simulates a campaign over a ground-truth raster, one round at a time. A round can be a day, a week or a year. `Campaign` draws and measures samples but chooses no allocation: a strategy chooses it. `score` turns the samples so far into one row per unit, `unit_stats` by default.


In [ ]:
class Campaign:
    "Simulated campaign over `fname_raster`, sampling the units of `gdf_units` one round at a time."
    def __init__(self,
        fname_raster:str, # Ground-truth raster
        gdf_units:gpd.GeoDataFrame, # Units indexed by `loc_id`
        score:callable=unit_stats, # Takes samples, units and raster; returns one row per unit
    ):
        store_attr()
        self.sampler, self.collector = Sampler(gdf_units), DataCollector(fname_raster)


`reset` starts a campaign with no samples. It also seeds NumPy's global random state, which `MoranICB` uses for its permutation test because `esda.Moran` takes no seed:

In [ ]:
@patch
def reset(self:Campaign,
    seed:int=None, # Seed for sampling, strategies and `MoranICB`'s permutation test
) -> gpd.GeoDataFrame: # The samples so far: none
    "Start a new campaign with no samples."
    self.rng, self.rounds = np.random.default_rng(seed), []
    np.random.seed(seed)
    return self.samples

`samples` holds the measurements of every round so far, with the round each one was taken in:

In [ ]:
@patch(as_prop=True)
def samples(self:Campaign) -> gpd.GeoDataFrame: # Measurements indexed by `loc_id`, with `value` and `round` columns
    if not self.rounds: return gpd.GeoDataFrame(index=pd.Index([], name='loc_id'), geometry=None, data={'value': None, 'round': None})
    return pd.concat(self.rounds)

`step` draws and measures the samples of one round, then scores all samples so far with `score`:

In [ ]:
@patch
def step(self:Campaign,
    n:np.ndarray, # Number of samples for each unit, in `gdf_units` order
) -> pd.DataFrame: # `score` of all samples so far
    "Draw and measure `n` samples per unit, then score all samples so far."
    pts = self.sampler.sample(n, method='uniform', rng=self.rng)
    self.rounds.append(self.collector.collect(pts).assign(round=len(self.rounds) + 1))
    return self.score(self.samples, self.gdf_units, self.fname_raster)

A strategy is a function that takes the samples so far, the units, the round's budget and the campaign's random generator, and returns the number of samples for each unit. `mcdm` ranks the units by MCDM on the criteria that `cbs` computes and splits the budget with `rank_to_sample`. While there are no samples, it ranks from the prior alone. The prior is the ground truth itself until #11 adds a degraded prior. The defaults give the strategy of [Run a campaign over several years](how-to/01_multi_year_campaign.ipynb):


In [ ]:
def mcdm(
    fname_prior:str, # Raster used as the prior
    cbs:list=None, # Callbacks computing the criteria once there are samples. `None` means Max, Min, Std, Count, Moran's I and Prior
    is_benefit:list=[True, True, True, False, False, True], # Benefit (`True`) or cost (`False`) flag of each criterion
    weights:list=[0.2, 0.1, 0.1, 0.2, 0.2, 0.2], # Weight of each criterion
    policy:str='quantiles', # Allocation policy of `rank_to_sample`
    min:int=1, # Minimum number of samples per unit
):
    "Strategy ranking units by MCDM, from the prior alone while there are no samples."
    cbs = cbs or [MaxCB(), MinCB(), StdCB(), CountCB(), MoranICB(k=5), PriorCB(fname_prior)]
    def strategy(samples, gdf_units, budget, rng):
        if samples.empty: cbs_, isb, w = [PriorCB(fname_prior)], [True], [1]
        else: cbs_, isb, w = cbs, is_benefit, weights
        state = State(samples, gdf_units, cbs=cbs_)()
        ranks = Optimizer(state=state).get_rank(is_benefit_x=isb, w_vector=w, n_method='LINEAR1', s_method='CP')
        return rank_to_sample(ranks['rank'].sort_index().values, budget=budget, min=min, policy=policy)
    return strategy


`run` plays a strategy for one budget per round and returns the per-unit tables of every round, indexed by `round` and `loc_id`:

In [ ]:
def run(
    campaign:Campaign, # Campaign to play
    strategy:callable, # Takes samples so far, units, budget and random generator; returns samples per unit
    budgets:list, # Budget of each round
    seed:int=None, # Seed passed to `Campaign.reset`
) -> pd.DataFrame: # `unit_stats` after each round, indexed by `round` and `loc_id`
    "Play `strategy` on `campaign` for `len(budgets)` rounds."
    campaign.reset(seed)
    tables = {r: campaign.step(strategy(campaign.samples, campaign.gdf_units, b, campaign.rng)) for r, b in enumerate(budgets, 1)}
    return pd.concat(tables, names=['round'])

Six rounds of 150 samples with the how-to strategy. The same seed gives the same campaign:

In [ ]:
campaign = Campaign(fname_raster, gdf_grid)
res = run(campaign, mcdm(fname_raster), budgets=[150] * 6, seed=0)
test_eq(res.loc[6].n.sum(), len(campaign.samples))
test_eq(campaign.samples['round'].max(), 6)
test_eq(run(campaign, mcdm(fname_raster), budgets=[150] * 6, seed=0), res)

`summarise` averages the errors over units after each round:

In [ ]:
def summarise(res): 
    "Mean over units of the errors in each round of `res`."
    return res.assign(abs_err_mean=(res.meas_mean - res.true_mean).abs(), err_max=res.meas_max - res.true_max
        ).groupby('round').agg(samples=('n', 'sum'), unsampled=('n', lambda n: (n == 0).sum()), mean_abs_err_mean=('abs_err_mean', 'mean'),
                               mean_err_max=('err_max', 'mean'), mean_wasserstein=('wasserstein', 'mean'))

summarise(res).round(4)

## Baseline strategies

The baselines place the MCDM strategies between allocations that need no ranking. `allocate` gives each unit `min` samples, then splits the rest of the budget in proportion to a weight per unit. It rounds each share down and gives the samples left over to the units with the largest remainders. A random amount far below one sample breaks ties between equal remainders:

In [ ]:
def allocate(
    weights:np.ndarray, # Weight of each unit
    budget:int, # Total number of samples
    min:int=1, # Minimum number of samples per unit
    rng:np.random.Generator=None, # Generator breaking ties between equal remainders
) -> np.ndarray: # Number of samples for each unit, summing to `budget`
    "Split `budget` across units in proportion to `weights`, after `min` samples each."
    weights = np.asarray(weights, dtype=float)
    if weights.sum() == 0: weights = np.ones_like(weights)
    n = np.full(len(weights), min)
    if budget < n.sum(): raise ValueError(f"`budget` {budget} is below {n.sum()}, the minimum over all units")
    share = (budget - n.sum()) * weights / weights.sum()
    n += np.floor(share).astype(int)
    rem = share - np.floor(share) + np.random.default_rng(rng).random(len(share)) * 1e-9
    n[np.argsort(-rem)[:budget - n.sum()]] += 1
    return n

In [ ]:
n = allocate([1, 1, 1, 3], 10, rng=0)
test_eq(n.sum(), 10)
n

`equal` gives every unit the same number of samples, up to one sample of rounding. Every other strategy should do at least as well:

In [ ]:
def equal(
    min:int=1, # Minimum number of samples per unit in each round
):
    "Strategy giving every unit the same share of the budget."
    def strategy(samples, gdf_units, budget, rng): return allocate(np.ones(len(gdf_units)), budget, min=min, rng=rng)
    return strategy


The textbook allocation for estimating each unit's mean gives a unit samples in proportion to its standard deviation. On units of equal size this is Neyman allocation. `neyman` estimates the standard deviation from the samples so far. A unit with fewer than 2 samples gets the largest estimate, and the first round, with no samples at all, is equal allocation. `oracle` uses the true standard deviation of each unit from the ground truth. No real campaign can do this, so it is an upper reference. Both split each round's budget on its own, without accounting for the samples of earlier rounds:

In [ ]:
def neyman(
    min:int=1, # Minimum number of samples per unit in each round
):
    "Strategy allocating in proportion to each unit's standard deviation, estimated from the samples so far."
    def strategy(samples, gdf_units, budget, rng):
        if samples.empty: return allocate(np.ones(len(gdf_units)), budget, min=min, rng=rng)
        std = samples.value.astype(float).groupby(level=0).std().reindex(gdf_units.index)
        return allocate(std.fillna(std.max()).values, budget, min=min, rng=rng)
    return strategy

def oracle(
    fname_raster:str, # Ground-truth raster
    min:int=1, # Minimum number of samples per unit in each round
):
    "Strategy allocating in proportion to each unit's true standard deviation."
    std = {}
    def strategy(samples, gdf_units, budget, rng):
        for loc_id, geom in gdf_units.geometry.items():
            if loc_id not in std: std[loc_id] = zonal_values(fname_raster, geom).std()
        return allocate(gdf_units.index.map(std).values, budget, min=min, rng=rng)
    return strategy


## Comparing strategies

`compare` runs each strategy with each seed and keeps the `summarise` table of every run. Every strategy gives each unit at least one sample per round, as the how-to strategy does:


In [ ]:
def compare(
    campaign:Campaign, # Campaign to play
    strategies:dict, # Strategies by name
    budgets:list, # Budget of each round
    seeds:list, # One run per seed
    summary:callable=summarise, # Summarises one run's per-unit tables by round
) -> pd.DataFrame: # `summary` of every run, indexed by `strategy`, `seed` and `round`
    "Run every strategy in `strategies` once per seed."
    return pd.concat({(name, seed): summary(run(campaign, strategy, budgets, seed))
                      for name, strategy in strategies.items() for seed in seeds}, names=['strategy', 'seed'])

In [ ]:
strategies = {'equal': equal(), 'prior only': mcdm(fname_raster, [PriorCB(fname_raster)], [True], [1]),
              'max only': mcdm(fname_raster, [MaxCB()], [True], [1]), 'std only': mcdm(fname_raster, [StdCB()], [True], [1]),
              'how-to': mcdm(fname_raster), 'neyman': neyman(), 'oracle': oracle(fname_raster)}
cmp = compare(campaign, strategies, budgets=[150] * 6, seeds=range(5))


Mean Wasserstein distance over units, averaged over seeds, by round:

In [ ]:
cmp.groupby(['strategy', 'round']).mean_wasserstein.mean().unstack().round(4)

## Minimum number of samples

A minimum per round fixes part of every round's budget: with 100 units, `min=1` fixes 100 of 150 samples in each round, and 600 of 900 over six rounds. The reasons for a minimum concern the samples a unit has in total: each unit needs some measurement for its decision, `StdCB` and `MoranICB` need several samples, and a minimum protects against a wrong prior. `cumulative_min` tops up every unit with fewer than `k` samples so far, then lets the wrapped strategy target the rest of the budget. When the top-up needs the whole budget, the budget goes to the units in proportion to what they lack. The wrapped strategy does not see the round's top-up when it allocates:

In [ ]:
def cumulative_min(
    strategy:callable, # Strategy targeting the budget left after the top-up
    k:int, # Minimum number of samples per unit over the whole campaign
):
    "Strategy topping up every unit to `k` samples in total before `strategy` targets the rest of the budget."
    def wrapped(samples, gdf_units, budget, rng):
        have = samples.groupby(level=0).size().reindex(gdf_units.index, fill_value=0).values
        top = np.maximum(k - have, 0)
        if top.sum() >= budget: return allocate(top, budget, min=0, rng=rng)
        return top + strategy(samples, gdf_units, budget - top.sum(), rng)
    return wrapped

Five strategies under four minimums, three seeds each: no minimum, one sample per unit in every round, and one or two samples per unit over the whole campaign:

In [ ]:
factories = {'equal': equal, 'how-to': lambda m: mcdm(fname_raster, min=m),
             'std only': lambda m: mcdm(fname_raster, [StdCB()], [True], [1], min=m),
             'neyman': neyman, 'oracle': lambda m: oracle(fname_raster, min=m)}
minimums = {'none': lambda f: f(0), '1 per round': lambda f: f(1),
            '1 in total': lambda f: cumulative_min(f(0), 1), '2 in total': lambda f: cumulative_min(f(0), 2)}
cmp_min = pd.concat({name: compare(campaign, {k: make(f) for k, f in factories.items()}, budgets=[150] * 6, seeds=range(3))
                     for name, make in minimums.items()}, names=['minimum'])

Mean Wasserstein distance after round 6, averaged over seeds:

In [ ]:
cmp_min.xs(6, level='round').groupby(['strategy', 'minimum']).mean_wasserstein.mean().unstack()[list(minimums)].round(4)

Without a minimum per round, the oracle improves, and every strategy that estimates from the samples gets worse. `std only` is the extreme case: `StdCB` gives a unit with one sample a standard deviation of 0, and the strategy never chooses that unit again. With a minimum of 1 in total and seed 0, 25 units still have a single sample after six rounds.

## Decision score

The DSS4NAFA restrictions dashboard suggests an action for each unit from three settings: a permissible level, a minimum number of samples and an accepted proportion over the permissible level. `decide` applies that rule to a unit's values. A unit with no values is `undefined`, and one with fewer than `min_n` is `sample`. Otherwise the unit is restricted when the share of its values above `level` exceeds `accepted`. The dashboard's "Partially restrict" is left out:

In [ ]:
def decide(
    values:np.ndarray, # Values measured in one unit
    level:float, # Permissible level
    min_n:int=1, # Minimum number of values for a decision
    accepted:float=0.1, # Accepted proportion of values above `level`
) -> str: # `restrict`, `no restrict`, `sample` or `undefined`
    "Action the DSS4NAFA dashboard suggests for one unit."
    if len(values) == 0: return 'undefined'
    if len(values) < min_n: return 'sample'
    return 'restrict' if (values > level).mean() > accepted else 'no restrict'

In [ ]:
test_eq(decide(np.array([0.5, 0.9, 0.7]), level=0.8, accepted=0.1), 'restrict')
test_eq(decide(np.array([0.5, 0.9, 0.7]), level=0.8, accepted=0.5), 'no restrict')
test_eq(decide(np.array([0.9]), level=0.8, min_n=3), 'sample')

`decisions` applies the rule twice for each unit: to the samples, and to the raster cells, which give the true action. It is a `score` for `Campaign`:

In [ ]:
def decisions(
    samples:gpd.GeoDataFrame, # Measurements indexed by `loc_id`, with a `value` column
    gdf_units:gpd.GeoDataFrame, # Units indexed by `loc_id`
    fname_raster:str, # Ground-truth raster
    level:float=0.8, # Permissible level
    min_n:int=1, # Minimum number of samples for a decision
    accepted:float=0.1, # Accepted proportion of values above `level`
) -> pd.DataFrame: # One row per unit
    "True and suggested action of each unit, with the true and measured shares above `level`."
    rows = {}
    for loc_id, geom in gdf_units.geometry.items():
        true, meas = zonal_values(fname_raster, geom), samples.value[samples.index == loc_id].values.astype(float)
        rows[loc_id] = dict(n=len(meas), true_share=(true > level).mean(), share=(meas > level).mean() if len(meas) else np.nan,
                            true_action=decide(true, level, accepted=accepted), action=decide(meas, level, min_n, accepted))
    return pd.DataFrame.from_dict(rows, orient='index').rename_axis('loc_id')

`summarise_decisions` counts, after each round, the units restricted although the truth says no (`needless`), the units not restricted although the truth says restrict (`missed`), and the units left without a decision (`unresolved`):

In [ ]:
def summarise_decisions(res):
    "Counts of needless, missed and unresolved decisions in each round of `res`."
    return res.assign(needless=(res.action == 'restrict') & (res.true_action == 'no restrict'),
                      missed=(res.action == 'no restrict') & (res.true_action == 'restrict'),
                      unresolved=res.action.isin(['sample', 'undefined'])
                     ).groupby('round')[['needless', 'missed', 'unresolved']].sum().assign(wrong=lambda d: d.needless + d.missed)

With a permissible level of 0.8 and 10% accepted, 35 of the 100 units should be restricted. The raster is smooth: only 9 units have between 2% and 30% of their cells above the level, so few decisions are close. The strategies of the baseline comparison, one sample per unit per round, 5 seeds:

In [ ]:
dcampaign = Campaign(fname_raster, gdf_grid, score=decisions)
test_eq((dcampaign.score(campaign.samples.iloc[:0], gdf_grid, fname_raster).true_action == 'restrict').sum(), 35)
cmp_dec = compare(dcampaign, strategies, budgets=[150] * 6, seeds=range(5), summary=summarise_decisions)

Wrong decisions (needless plus missed), averaged over seeds, by round:

In [ ]:
cmp_dec.groupby(['strategy', 'round']).wrong.mean().unstack()

After six rounds, every strategy gets between 2.6 and 4.2 of the 100 decisions wrong, mostly missed restrictions, and the standard deviation over seeds reaches 2.7. Most differences are within that noise. The how-to strategy no longer trails equal allocation, and is ahead of it in rounds 2 and 3. With only 9 close units, this raster cannot separate the strategies. The `oracle` here targets the true standard deviation, not the decisions.